<a href="https://colab.research.google.com/github/zFonta/CEIA-TF-Chess-DL/blob/main/notebooks/09_jugar_contra_el_motor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 09 - Jugar contra el motor

Un tablero para **jugar** contra las redes entrenadas, acá adentro de la
notebook. No agrega ningún resultado a la memoria: todo lo que se mide está en
la [08](08_motor_y_partidas.ipynb). Lo que agrega es otra cosa —poder
**contradecir al modelo**— que para una defensa vale más que una tabla.

## Cómo se usa

**Click en la pieza, click en el casillero.** Los destinos legales aparecen
marcados con un punto; las capturas, con el casillero en rojo. Para enrocar
alcanza con clickear el rey y después la torre. Al coronar, el tablero pregunta
qué pieza querés en vez de asumir una dama.

## Qué mirar mientras jugás

| | |
|---|---|
| **Las dos barras** | La evaluación de la posición según **la red** y según **Stockfish**, en la misma escala y las dos desde las blancas. La diferencia entre ambas, sobre la posición que tenés delante, es un término del error que la memoria reporta promediado: el RMSE de test de 0,2511 es la raíz de la media de exactamente eso |
| **La escala** | `[-1, 1]` de la red y su equivalente en centipeones (requerimiento 4.2) |
| **"Lo que ve el motor"** | Las 5 jugadas mejor rankeadas **con su valor**. Es el criterio de la red, no sólo su decisión |
| **Profundidad** | 1 ply es el requerimiento 1.6; 2 es la extensión que en la 08 partió al medio la pérdida en centipeones |
| **Modelo** | ResNet y Transformer empataron por RMSE y volvieron a empatar en pérdida a 2 plies. Cambiar de una a otra en la misma posición es la única forma de tener alguna intuición de qué significa ese empate |

> **Sobre la fuerza del motor.** En la 08 la escalera de Elo lo ubicó cerca de
> un rival débil de Stockfish. Es un motor de evaluación estática con una
> búsqueda de uno o dos plies: no hay poda alfa-beta, ni tablas de
> transposición, ni búsqueda de capturas. Va a jugar aperturas razonables y a
> colgar piezas en tácticas de tres jugadas, y eso es exactamente lo que
> predicen los números del bloque 6.

## 1. Entorno

In [ ]:
# Clonar el repositorio e instalar el paquete.
# Idempotente: se puede volver a correr tal cual despues de una desconexion.
import os, sys, subprocess, importlib
from pathlib import Path

REPO_DIR = Path("/content/CEIA-TF-Chess-DL")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "https://github.com/zFonta/CEIA-TF-Chess-DL.git", str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"], check=True)

# pip registra el paquete editable con un .pth, y los .pth solo se procesan al
# arrancar el interprete: un kernel que ya esta corriendo no lo ve. Agregar src/
# a sys.path lo hace visible sin tener que reiniciar el runtime.
src = str(REPO_DIR / "src")
if src not in sys.path:
    sys.path.insert(0, src)
importlib.invalidate_caches()

# Stockfish con version fija. Entrenar no lo usa, pero sin el se saltean los 17
# tests de integracion del pipeline, que son la evidencia del requerimiento 3.2.
subprocess.run(["bash", "scripts/setup_stockfish.sh"], check=True)
os.environ["PATH"] = f"{REPO_DIR}/bin:" + os.environ["PATH"]

print("Listo. Directorio de trabajo:", os.getcwd())

In [ ]:
import torch

dispositivo = 'cuda' if torch.cuda.is_available() else 'cpu'
print('GPU  :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'sin GPU')
print('Se usa:', dispositivo)

# Esta notebook anda igual en CPU: una jugada a profundidad 1 evalua unas 30
# posiciones y a profundidad 2 unas 1.000, que en CPU son decimas de segundo.
# El runtime con GPU esta bien igual, pero no hace falta pedirlo para jugar.

## 2. Los modelos

Los mismos checkpoints que jugaron el torneo de la 08: el mejor por validación
de cada arquitectura. No hace falta bajar el dataset — el tablero sólo necesita
las redes, así que esta notebook arranca en menos de un minuto.

In [ ]:
import chess.engine

from chessdl import hf
from chessdl.config import load_config
from chessdl.engine.evaluator import Evaluator
from chessdl.engine.loader import load_from_hub

cfg = load_config()
token = hf.get_token()
repo_modelos = f'{cfg.output.hf_namespace}/{cfg.training.hf_models_repo}'

CORRIDAS = {
    'ResNet':      'campana2-warmup',
    'Transformer': 'transformer-campana2-lotes-chicos',
}

evaluadores = {}
for nombre, corrida in CORRIDAS.items():
    modelo = load_from_hub(repo_modelos, corrida, token=token, device=dispositivo)
    evaluadores[nombre] = Evaluator(modelo, device=dispositivo)
    print(f'{nombre:<12}{evaluadores[nombre].describe()}')

# Stockfish como referencia, no como rival: el tablero lo consulta para mostrar
# su lectura al lado de la de la red. La celda de entorno ya lo instalo.
referencia = chess.engine.SimpleEngine.popen_uci('stockfish')
print('referencia  ', referencia.id.get('name', 'Stockfish'))

## 3. El tablero

Los botones de posición cargan casos elegidos, y cada uno muestra algo que la
08 midió:

- **Mate en 1** — el motor lo encuentra siempre, a cualquier profundidad. No
  porque la red lo evalúe bien (nunca vio un mate: el dataset excluyó las
  posiciones terminales) sino porque las posiciones decididas se resuelven por
  reglas antes de llegar a la red. Es una de las dos protecciones del bloque 5,
  y acá se ve funcionando.
- **Dama colgada** — el punto ciego de la profundidad 1. `Qxd5` gana un peón
  defendido por `c6`: a un ply el árbol termina en la jugada propia, así que la
  recaptura no está en el árbol y la jugada parece ganar material. **Mirá dónde
  aparece `Qxd5` en el ranking a profundidad 1 y a 2.** Ese salto es el que en
  la 08 bajó la pérdida media de 177 a 87 centipeones en la ResNet.
- **Mate en 1**, otra vez, pero mirando el panel: Stockfish anuncia el mate y
  la red informa una ventaja cualquiera. No es un error del modelo sino el
  limite de su escala — nunca vio un mate, porque el dataset excluyo las
  posiciones terminales. Que el motor igual lo juegue es merito de la busqueda,
  que las resuelve por reglas.
- **Final de torre** — rey y torre contra rey, ganado de manera trivial para
  cualquiera que sepa la técnica. El motor probablemente no lo gane, y el
  motivo está en el pipeline de datos: se muestrearon 4 posiciones por partida
  sin saltear aperturas, así que los finales quedaron sub-representados y la
  red nunca vio suficientes como para aprenderlos.

In [ ]:
from chessdl.ui import PlayUI

POSICIONES = {
    'Mate en 1':      '6k1/5ppp/8/8/8/8/8/R5K1 w - - 0 1',
    'Dama colgada':   'rnbqkbnr/pp3ppp/2p5/3p4/3Q4/8/PPP1PPPP/RNB1KBNR w KQkq - 0 1',
    'Final de torre': '8/8/8/5k2/8/8/4R3/4K3 w - - 0 1',
    'Inicial':        'rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1',
}

# `reference` es opcional: sin el, el panel muestra una sola barra. Con el,
# muestra las dos y su diferencia. La profundidad 12 no es arbitraria -- es la
# que etiqueto el dataset, asi que es la lectura que la red fue entrenada para
# reproducir. Una referencia mas profunda seria mejor ajedrecista y peor vara:
# la brecha mezclaria el error del modelo con la diferencia entre dos Stockfish,
# y de esos dos terminos solo el primero es de este trabajo.
tablero = PlayUI(evaluadores, depth=2, positions=POSICIONES,
                 reference=referencia, reference_depth=12)
tablero

## 4. Qué queda de esto

El tablero no produce ninguna métrica: las de la memoria salen de la
[08](08_motor_y_partidas.ipynb), donde cada número viene con su incertidumbre.
Lo que sí produce es la posibilidad de **verificar a mano** las tres cosas que
esa notebook afirma —que el mate se encuentra por reglas, que un ply no ve la
recaptura y que dos sí, y que las dos arquitecturas se parecen mucho más de lo
que sus nombres sugieren— sobre posiciones que elija quien esté leyendo.

**Próximo paso:** ninguno por ahora. El bloque 6 cierra el alcance de este
trabajo; la continuación —más posiciones etiquetadas, reentrenar las dos
arquitecturas sobre ese dataset y compararlas contra aprendizaje por refuerzo—
queda anotada como trabajo futuro en [`docs/modelado.md`](../docs/modelado.md).